# Curso 1: SQL Server – Nivel Básico
## Tema 2: Instalación y Configuración Inicial

### **Objetivo del Tema**
Al finalizar este tema, comprenderás el proceso de instalación de Microsoft SQL Server, dominarás los diferentes modos de autenticación (Windows y Mixta), la configuración de red y protocolos del motor, y sabrás distinguir cuándo emplear una instancia predeterminada frente a una instancia nombrada.

---

## 1. Planificación e Instalación de SQL Server

La instalación exitosa de SQL Server comienza con una fase de planificación. Configurar de forma incorrecta el servidor desde el inicio puede acarrear problemas graves de rendimiento y seguridad difíciles de corregir en producción.

### **1.1. Prácticas Clave en la Fase de Instalación**
*   **Separación de Unidades de Disco:** Nunca instales las bases de datos en la misma unidad que el sistema operativo (`C:\`). Se recomienda separar físicamente los discos de la siguiente forma:
    *   `Unidad D:` Binarios del sistema y bases de datos del sistema.
    *   `Unidad E:` Archivos de datos de bases de datos de usuario (`.mdf`).
    *   `Unidad F:` Archivos de registros de transacciones (`.ldf`).
    *   `Unidad G:` Archivo de la base de datos temporal (`tempdb`).
*   **Instant File Initialization (Inicialización Instantánea de Archivos):** Durante la instalación, se debe conceder el privilegio de realizar tareas de mantenimiento de volumen al servicio del motor de base de datos de SQL Server. Esto permite al motor omitir el llenado con ceros al crear o expandir archivos de datos, acelerando drásticamente operaciones como la creación de bases de datos y la restauración de copias de seguridad.
*   **Cuentas de Servicio (Service Accounts):** Se aconseja no ejecutar los servicios de SQL Server bajo la cuenta del sistema local (`LocalSystem`) ni bajo cuentas de administrador local. En su lugar, se deben usar **Cuentas de Servicio Administradas por el Grupo (gMSA)** o cuentas de usuario de dominio de privilegios mínimos para mitigar los riesgos en caso de vulnerabilidades en el servidor.

### **1.2 Formas de Instalación**
* Por Sistema Operativo
  * Windows
  * Linux
* Forma de instalación
  * Servidor Físico o Maquina Virtual
  * Docker (Contenedor)
* Tipo de Instalador
  * GUI
  * CMD

### **1.3 Setup GUI (acciones)**
- Nueva instalación de SQL Server.
- Agregar características a una instalación existente.
- Actualizar una instancia existente (Upgrade).
- Reparar una instalación.
- Agregar un nodo a un clúster de conmutación por error.
- Completar un nodo preparado (Sysprep).
- Instalar herramientas (según la versión).

_[Guia de Instalación](guia_instalacion_sql_server_2019.md)_

## 2. Modos de Autenticación: Windows vs. Mixta

SQL Server admite dos esquemas para controlar quién puede conectarse a la instancia:

### **2.1. Windows Authentication Mode (Autenticación Integrada)**
Es el modo por defecto y el recomendado por Microsoft por razones de seguridad. En este esquema:
*   SQL Server delega la validación de la identidad al sistema operativo Windows o a Active Directory (Kerberos o NTLM).
*   No se transmiten contraseñas por la red al conectarse; en su lugar, se emplean tokens de seguridad del dominio.
*   Se heredan las directivas corporativas de contraseñas (complejidad, caducidad y bloqueo de cuentas).

### **2.2. Mixed Mode (Modo Mixto)**
Habilita tanto la Autenticación de Windows como la Autenticación de SQL Server. En este modo:
*   Los usuarios de SQL Server se crean y se autentican de forma interna dentro del propio motor de base de datos.
*   Es necesario establecer una contraseña robusta para la cuenta predeterminada de superusuario `sa` (System Administrator).
*   Es muy útil cuando las aplicaciones que consumen la base de datos se ejecutan en sistemas operativos diferentes a Windows (como Linux o macOS) o cuando los servidores no forman parte de un dominio de Active Directory corporativo.

### **Laboratorio 2.1: Diagnóstico del Modo de Autenticación**
Ejecuta la siguiente consulta para verificar si tu servidor se encuentra configurado en Autenticación de Windows únicamente o en Modo Mixto.

In [10]:
-- Consultar el modo de seguridad de la instancia
-- Retorna 1 para Autenticación de Windows
-- Retorna 0 para Modo Mixto
SELECT 
    SERVERPROPERTY('IsIntegratedSecurityOnly') AS [Es_Solo_Autenticacion_Windows],
    CASE SERVERPROPERTY('IsIntegratedSecurityOnly')
        WHEN 1 THEN 'Autenticación de Windows (Solo)'
        WHEN 0 THEN 'Modo Mixto (Windows y SQL Server)'
    END AS [Modo_Seguridad_Activo];

(1 row affected)

Es_Solo_Autenticacion_Windows | Modo_Seguridad_Activo            
------------------------------+----------------------------------
0                             | Modo Mixto (Windows y SQL Server)
(1 row)

Total execution time: 00:00:00.065

---
## 3. Configuración de Red y Protocolos

Para que los clientes se conecten a SQL Server desde la red, es necesario activar y configurar los protocolos en el **SQL Server Configuration Manager**.

### **3.1. Protocolos de Red de SQL Server**
1.  **Shared Memory (Memoria Compartida):** Es el protocolo más rápido y se utiliza automáticamente cuando el cliente de SSMS y el motor se ejecutan en la misma máquina física. No funciona para conexiones de red.
2.  **TCP/IP:** Es el protocolo de red estándar y obligatorio para permitir que aplicaciones e interfaces remotas accedan al servidor SQL Server.
3.  **Named Pipes (Canales con Nombre):** Protocolo diseñado para redes de área local (LAN). Es alternativo a TCP/IP y suele desactivarse por razones de simplicidad y seguridad.

### **3.2. Asignación de Puertos TCP**
*   **Puerto Estándar (TCP 1433):** La instancia predeterminada de SQL Server escucha por defecto en el puerto TCP 1433. En entornos de alta seguridad, se cambia este puerto a uno no estándar (por ejemplo, TCP 57211) para ocultar el motor de los escaneos automáticos de puertos en la red corporativa.
*   **SQL Server Browser (UDP 1434):** Cuando se utilizan múltiples instancias en un mismo servidor físico, éstas escuchan en puertos dinámicos. SQL Server Browser ayuda a los clientes externos a descubrir dinámicamente en qué puerto está escuchando cada instancia nombrada.

### **Laboratorio 2.2: Consultar Protocolos y Puertos Activos**
Utiliza estas consultas avanzadas para identificar los puertos TCP en los que SQL Server está respondiendo activamente.

In [12]:
-- 1. Consultar los listeners TCP habilitados en el motor
-- Nota: Requiere permiso de servidor 'VIEW SERVER STATE'.
SELECT 
    listener_id,
    ip_address AS [IP_Escucha],
    port AS [Puerto_TCP],
    CASE is_ipv4 
        WHEN 1 THEN 'IPv4' 
        ELSE 'IPv6' 
    END AS [Protocolo_IP],
    state_desc AS [Estado_Listener]
FROM sys.dm_tcp_listener_states;

(6 rows affected)

listener_id | IP_Escucha | Puerto_TCP | Protocolo_IP | Estado_Listener
------------+------------+------------+--------------+----------------
5           | ::1        | 49682      | IPv6         | ONLINE         
6           | 127.0.0.1  | 49682      | IPv4         | ONLINE         
3           | ::1        | 1434       | IPv6         | ONLINE         
4           | 127.0.0.1  | 1434       | IPv4         | ONLINE         
1           | ::         | 1433       | IPv6         | ONLINE         
2           | 0.0.0.0    | 1433       | IPv4         | ONLINE         
(6 rows)

Total execution time: 00:00:00.063

In [13]:
-- 2. Alternativa: Leer el registro de errores para ver en qué puerto TCP escucha el motor
EXEC xp_readerrorlog 0, 1, N'Server is listening on';

(6 rows affected)

LogDate                 | ProcessInfo | Text                                             
------------------------+-------------+--------------------------------------------------
2026-06-10 22:00:24.770 | spid22s     | Server is listening on [ 'any' <ipv6> 1433].     
2026-06-10 22:00:24.780 | spid22s     | Server is listening on [ 'any' <ipv4> 1433].     
2026-06-10 22:00:24.780 | Server      | Server is listening on [ ::1 <ipv6> 1434].       
2026-06-10 22:00:24.790 | Server      | Server is listening on [ 127.0.0.1 <ipv4> 1434]. 
2026-06-10 22:00:24.820 | spid22s     | Server is listening on [ ::1 <ipv6> 49682].      
2026-06-10 22:00:24.820 | spid22s     | Server is listening on [ 127.0.0.1 <ipv4> 49682].
(6 rows)

Total execution time: 00:00:00.975

---
## 4. Instancias Predeterminadas vs. Instancias Nombradas

SQL Server permite realizar múltiples instalaciones independientes del motor de base de datos en una misma máquina física o máquina virtual. Cada una de estas instalaciones se conoce como una **Instancia**.

| Aspecto | **Instancia Predeterminada (Default Instance)** | **Instancia Nombrada (Named Instance)** |
| :--- | :--- | :--- |
| **Cantidad Máxima** | Máximo **1** por máquina física o virtual. | Hasta **50** en un mismo servidor. |
| **Nomenclatura** | Adopta únicamente el nombre del servidor (ej. `SERVIDOR-PROD`). | Nombre del servidor seguido de una barra diagonal inversa y un identificador (ej. `SERVIDOR-PROD\DESARROLLO`). |
| **Puerto por Defecto** | TCP 1433 fijo. | Puertos dinámicos asignados por el S.O. durante el inicio (requiere servicio Browser). |
| **Uso Sugerido** | Para el motor principal del servidor corporativo. | Para aislar bases de datos de desarrollo, pruebas, o de diferentes clientes en un mismo hardware. |

Cada instancia funciona como un servidor de bases de datos completamente aislado: tiene sus propios archivos del sistema (`master`, `model`, `msdb`, `tempdb`), su propia asignación de RAM, su propia configuración de seguridad de logins y sus propios trabajos del agente.

### **Laboratorio 2.3: Consulta del Servidor e Instancia**
Utiliza la siguiente consulta para obtener los detalles específicos del nombre del servidor físico y de la instancia activa con la que estás trabajando.

In [14]:
-- Obtener detalles de la máquina y la instancia actual
SELECT 
    CAST(SERVERPROPERTY('MachineName') AS VARCHAR(100)) AS [Servidor_Fisico],
    COALESCE(CAST(SERVERPROPERTY('InstanceName') AS VARCHAR(100)), 'INSTANCIA PREDETERMINADA') AS [Instancia_SQL],
    CAST(SERVERPROPERTY('ServerName') AS VARCHAR(100)) AS [Nombre_Conexion_Servidor_Completo];

(1 row affected)

Servidor_Fisico | Instancia_SQL            | Nombre_Conexion_Servidor_Completo
----------------+--------------------------+----------------------------------
4ECF09D         | INSTANCIA PREDETERMINADA | 4ECF09D                          
(1 row)

Total execution time: 00:00:00.084

---
## 5. Autoevaluación y Ejercicios Prácticos del Tema 2

Valida lo aprendido en esta sección respondiendo las siguientes preguntas teóricas:

### **Cuestionario de Autoevaluación**

1. **¿Por qué se recomienda habilitar la opción 'Instant File Initialization' durante el proceso de instalación de SQL Server?**
   * *Respuesta:* Permite que el motor de base de datos asigne y extienda espacio físico en disco para los archivos de datos sin necesidad de inicializarlos con ceros, lo que acelera en gran medida la creación de bases de datos de gran tamaño y los procesos de restauración.

2. **¿Cuál es la principal ventaja de seguridad de utilizar la Autenticación de Windows en lugar de la Autenticación de SQL Server?**
   * *Respuesta:* La Autenticación de Windows delega la seguridad a Active Directory o al S.O., evitando la transmisión de contraseñas en texto claro a través de la red y aplicando de forma nativa políticas complejas como la caducidad y el bloqueo de cuentas corporativas.

3. **Si configuras una instancia nombrada llamada `DBServer\Ventas`, ¿qué servicio de SQL Server es necesario que esté activo para que los clientes remotos puedan descubrir el puerto dinámico de conexión de esta instancia?**
   * *Respuesta:* El servicio **SQL Server Browser** (que escucha de forma persistente en el puerto UDP 1434).

4. **¿Cuál es el puerto TCP estándar reservado para una instancia predeterminada de SQL Server?**
   * *Respuesta:* El puerto **TCP 1433**.

5. **¿Qué sucede si instalas dos instancias distintas en el mismo servidor físico con respecto a la memoria RAM?**
   * *Respuesta:* Cada instancia se ejecuta en un proceso de sistema operativo aislado. Ambas competirán por los recursos de memoria física de la máquina. Por tanto, se recomienda configurar el parámetro `max server memory` de manera explícita en cada una de ellas para evitar que una instancia consuma toda la RAM y desestabilice a la otra.

---
### **Práctica de Laboratorio Sugerida**
1. **Configuración de Red:** Abre la herramienta **SQL Server Configuration Manager** en el servidor de base de datos. Navega a `SQL Server Network Configuration` -> `Protocols for MSSQLSERVER` (o el nombre de tu instancia). Verifica que `TCP/IP` esté en estado 'Enabled'.
2. **Configuración de Puerto:** Haz doble clic en `TCP/IP`, ve a la pestaña `IP Addresses` y verifica el puerto asignado en la propiedad `TCP Port` al final de la lista.
3. **Ejecución y Verificación:** Abre este notebook y ejecuta las consultas de diagnóstico para confirmar los puertos TCP de escucha activos y corroborar el modo de autenticación configurado.